# 4\.1\.1 Construct Forecasting Target Sequences and Horizons

This notebook constructs the canonical target sequences for the forecasting pipeline. It reshapes five selected mobility metrics—Taxi demand, Taxi speed, FHVHV demand, FHVHV speed, and Subway ridership—into ordered Taxi Zone × Metric sequences at daypart resolution. These targets provide complementary demand and operating-condition signals across the project's principal observational mobility modes. The notebook also carries forward the series-support evidence established in `3.3.1` and evaluates whether the unified sequences have sufficient continuity and coverage for forecasting. The first stage stops before assigning final forecasting eligibility so mixed daypart-support patterns can be reviewed before a forecasting rule is locked.

Load the libraries and define the canonical inputs, intermediate paths, recomputation toggles, forecasting targets, study dates, temporal ordering, and reusable validation helpers used throughout the notebook. The `3.3.1` output is used only to recover established series-support provenance; its residual and fitted decomposition values are not forecasting inputs.

In [1]:
# ---------------------------------------------------------------------
# 4.1.1.0 — Notebook setup
# ---------------------------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 180)

PIPELINE_DATA_DIR = Path("pipeline_data")
INPUT_131_DIR = PIPELINE_DATA_DIR / "1.3.1.final_tables"
INPUT_331_DIR = PIPELINE_DATA_DIR / "3.3.1.final_tables"
INTERMEDIATE_411_DIR = PIPELINE_DATA_DIR / "4.1.1.intermediate_tables"
FINAL_411_DIR = PIPELINE_DATA_DIR / "4.1.1.final_tables"

INTERMEDIATE_411_DIR.mkdir(parents=True, exist_ok=True)
FINAL_411_DIR.mkdir(parents=True, exist_ok=True)

ANALYSIS_READY_PANEL_PATH = INPUT_131_DIR / "analysis_ready_mobility_panel.parquet"
RESIDUAL_SUPPORT_PANEL_PATH = INPUT_331_DIR / "temporal_residual_mobility_panel.parquet"
RESIDUAL_SUPPORT_DIAGNOSTICS_PATH = INPUT_331_DIR / "temporal_residual_mobility_diagnostics.parquet"
FORECAST_SEQUENCE_SUPPORT_DIAGNOSTICS_PATH = (
    INTERMEDIATE_411_DIR / "forecast_sequence_support_diagnostics.parquet"
)

REBUILD_FORECAST_SEQUENCE_SUPPORT_DIAGNOSTICS = False
RUN_FORECAST_SEQUENCE_QA = True

STUDY_START_DATE = pd.Timestamp("2023-01-01")
STUDY_END_DATE = pd.Timestamp("2026-03-31")
CONGESTION_PRICING_START_DATE = pd.Timestamp("2025-01-05")
FORECAST_DEVELOPMENT_START_DATE = CONGESTION_PRICING_START_DATE
FORECAST_DEVELOPMENT_END_DATE = pd.Timestamp("2026-01-04")
FORECAST_EVALUATION_START_DATE = pd.Timestamp("2026-01-05")
FORECAST_EVALUATION_END_DATE = STUDY_END_DATE

FORECAST_TARGET_METRICS = [
    "taxi_trip_count",
    "taxi_avg_trip_speed",
    "fhvhv_trip_count",
    "fhvhv_avg_trip_speed",
    "subway_ridership",
]

DAYPART_ORDER = [
    "overnight",
    "am_peak",
    "midday",
    "pm_peak",
    "evening",
]

TEMPORAL_BUCKET_ORDER = [
    "weekday_overnight",
    "weekday_am_peak",
    "weekday_midday",
    "weekday_pm_peak",
    "weekday_evening",
    "weekend_overnight",
    "weekend_am_peak",
    "weekend_midday",
    "weekend_pm_peak",
    "weekend_evening",
]

PLANNED_FORECAST_HORIZONS = (1, 2, 5)

SEQUENCE_PANEL_ID_COLUMNS = [
    "taxi_zone_id",
    "zone",
    "borough",
    "canonical_location_id",
    "date",
    "daypart",
    "daypart_order",
    "temporal_bucket",
    "temporal_bucket_order",
    "observation_sequence_id",
    "pre_post_cp",
]

PANEL_GRAIN_COLUMNS = ["taxi_zone_id", "date", "temporal_bucket"]
LONG_SEQUENCE_GRAIN_COLUMNS = [
    "taxi_zone_id",
    "metric",
    "observation_sequence_id",
]
FORECAST_SEQUENCE_ID_COLUMNS = ["taxi_zone_id", "metric"]
DAYPART_SUPPORT_GRAIN_COLUMNS = ["taxi_zone_id", "metric", "daypart"]


def require_columns(frame: pd.DataFrame, required_columns: list[str], frame_name: str) -> None:
    """Raise an explicit error when a required column is missing."""
    missing_columns = [column for column in required_columns if column not in frame.columns]
    assert not missing_columns, f"{frame_name} is missing required columns: {missing_columns}"



def duplicate_count(frame: pd.DataFrame, key_columns: list[str]) -> int:
    """Return the number of duplicate rows at the requested grain."""
    return int(frame.duplicated(subset=key_columns).sum())



def compact_column_groups(column_groups: dict[str, list[str]]) -> pd.DataFrame:
    """Render grouped column names in a compact reader-facing table."""
    return pd.DataFrame(
        [
            {
                "Column group": group_name,
                "Columns": ", ".join(columns),
            }
            for group_name, columns in column_groups.items()
        ]
    )



def classify_support_pattern(modeled_dayparts: int, fallback_dayparts: int, unavailable_dayparts: int, represented_dayparts: int) -> str:
    """Describe the daypart-support mixture without implying forecast eligibility."""
    if represented_dayparts == 0:
        return "support provenance missing"
    if unavailable_dayparts == represented_dayparts:
        return "all unavailable"
    if modeled_dayparts == represented_dayparts:
        return "all modeled"
    if fallback_dayparts == represented_dayparts:
        return "all fallback"
    if unavailable_dayparts > 0:
        return "contains unavailable"
    if modeled_dayparts > 0 and fallback_dayparts > 0:
        return "modeled + fallback"
    if modeled_dayparts > 0:
        return "modeled only subset"
    if fallback_dayparts > 0:
        return "fallback only subset"
    return "mixed support"



def longest_true_run(mask_like: pd.Series | np.ndarray) -> int:
    """Return the longest consecutive run of True values in sequence order."""
    mask = np.asarray(mask_like, dtype=bool)
    longest_run = 0
    current_run = 0

    for is_true in mask:
        current_run = current_run + 1 if is_true else 0
        longest_run = max(longest_run, current_run)

    return int(longest_run)



def count_false_runs(mask_like: pd.Series | np.ndarray) -> int:
    """Count consecutive missing-observation runs without dropping rows first."""
    mask = np.asarray(mask_like, dtype=bool)
    false_runs = 0
    in_false_run = False

    for is_true in mask:
        if is_true:
            in_false_run = False
            continue
        if not in_false_run:
            false_runs += 1
            in_false_run = True

    return int(false_runs)



def max_false_run_length(mask_like: pd.Series | np.ndarray) -> int:
    """Return the maximum length of any consecutive missing-observation run."""
    mask = np.asarray(mask_like, dtype=bool)
    longest_run = 0
    current_run = 0

    for is_true in mask:
        current_run = 0 if is_true else current_run + 1
        longest_run = max(longest_run, current_run)

    return int(longest_run)



def round_display(frame: pd.DataFrame, digits: int = 3) -> pd.DataFrame:
    """Round numeric display values without mutating the analytical source object."""
    rounded_frame = frame.copy()
    numeric_columns = rounded_frame.select_dtypes(include=[np.number]).columns
    rounded_frame[numeric_columns] = rounded_frame[numeric_columns].round(digits)
    return rounded_frame


print(f"Canonical mobility panel: {ANALYSIS_READY_PANEL_PATH}")
print(f"3.3.1 support provenance panel: {RESIDUAL_SUPPORT_PANEL_PATH}")
print(f"4.1.1 intermediate directory: {INTERMEDIATE_411_DIR}")
print(f"4.1.1 final directory defined for later use: {FINAL_411_DIR}")
print(f"Forecast targets: {FORECAST_TARGET_METRICS}")
print(f"Planned horizons (reference only): {PLANNED_FORECAST_HORIZONS}")

Canonical mobility panel: pipeline_data/1.3.1.final_tables/analysis_ready_mobility_panel.parquet
3.3.1 support provenance panel: pipeline_data/3.3.1.final_tables/temporal_residual_mobility_panel.parquet
4.1.1 intermediate directory: pipeline_data/4.1.1.intermediate_tables
4.1.1 final directory defined for later use: pipeline_data/4.1.1.final_tables
Forecast targets: ['taxi_trip_count', 'taxi_avg_trip_speed', 'fhvhv_trip_count', 'fhvhv_avg_trip_speed', 'subway_ridership']
Planned horizons (reference only): (1, 2, 5)


## 4.1.1.1 Validate Canonical Forecasting Inputs

This chapter establishes the source contract for forecasting-sequence construction. It loads the canonical 1.3.1 mobility panel and confirms that the required spatial, temporal, sequence-ordering, and target fields are present at the expected Taxi Zone × Date × Temporal Bucket grain. The five selected forecasting targets—Taxi demand, Taxi speed, FHVHV demand, FHVHV speed, and Subway ridership—are retained because they provide complementary demand and operating-condition signals across the project's principal observational mobility modes.

### Load and validate the canonical mobility panel

Load only the canonical identifiers, temporal-ordering fields, policy-period metadata, and five forecasting targets needed for target-sequence construction, then validate the source contract before reshaping.

In [2]:
# ---------------------------------------------------------------------
# 4.1.1.1A — Load and validate the canonical mobility panel
# ---------------------------------------------------------------------

required_input_paths = {
    "analysis_ready_mobility_panel": ANALYSIS_READY_PANEL_PATH,
    "3.3.1_support_panel": RESIDUAL_SUPPORT_PANEL_PATH,
}

path_status_df = pd.DataFrame(
    [
        {
            "Asset": asset_name,
            "Path": str(asset_path),
            "Exists": asset_path.exists(),
        }
        for asset_name, asset_path in required_input_paths.items()
    ]
)

assert path_status_df["Exists"].all(), "A required upstream forecasting input is missing."

panel_columns_to_load = SEQUENCE_PANEL_ID_COLUMNS + FORECAST_TARGET_METRICS
mobility_panel_df = pd.read_parquet(
    ANALYSIS_READY_PANEL_PATH,
    columns=panel_columns_to_load,
)

# Normalize the canonical temporal fields but do not reconstruct anything that is missing.
mobility_panel_df["date"] = pd.to_datetime(mobility_panel_df["date"])
require_columns(mobility_panel_df, panel_columns_to_load, "mobility_panel_df")

observed_daypart_cycle = (
    mobility_panel_df[["daypart", "daypart_order"]]
    .drop_duplicates()
    .sort_values(["daypart_order", "daypart"])
    ["daypart"]
    .tolist()
)

observed_temporal_bucket_cycle = (
    mobility_panel_df[["temporal_bucket", "temporal_bucket_order"]]
    .drop_duplicates()
    .sort_values(["temporal_bucket_order", "temporal_bucket"])
    ["temporal_bucket"]
    .tolist()
)

source_duplicate_rows = duplicate_count(
    mobility_panel_df,
    PANEL_GRAIN_COLUMNS,
)

source_contract_df = pd.DataFrame(
    [
        {
            "Rows": len(mobility_panel_df),
            "Taxi Zones": mobility_panel_df["taxi_zone_id"].nunique(),
            "First date": mobility_panel_df["date"].min(),
            "Last date": mobility_panel_df["date"].max(),
            "Dayparts": mobility_panel_df["daypart"].nunique(),
            "Temporal buckets": mobility_panel_df["temporal_bucket"].nunique(),
            "Forecast targets": len(FORECAST_TARGET_METRICS),
            "Duplicate canonical rows": source_duplicate_rows,
            "Missing daypart_order": int(mobility_panel_df["daypart_order"].isna().sum()),
            "Missing temporal_bucket_order": int(mobility_panel_df["temporal_bucket_order"].isna().sum()),
            "Missing observation_sequence_id": int(mobility_panel_df["observation_sequence_id"].isna().sum()),
        }
    ]
)

loaded_target_columns_df = pd.DataFrame(
    {
        "Loaded forecast target columns": FORECAST_TARGET_METRICS,
    }
)

if RUN_FORECAST_SEQUENCE_QA:
    assert mobility_panel_df["date"].between(STUDY_START_DATE, STUDY_END_DATE).all(), (
        "The canonical mobility panel extends outside the 4.1.1 study window."
    )
    assert source_duplicate_rows == 0, (
        "The canonical mobility panel is not unique at Taxi Zone × Date × Temporal Bucket grain."
    )
    assert observed_daypart_cycle == DAYPART_ORDER, (
        f"Observed daypart order differs from the expected canonical order: {observed_daypart_cycle}"
    )
    assert observed_temporal_bucket_cycle == TEMPORAL_BUCKET_ORDER, (
        "Observed temporal_bucket order differs from the expected canonical order."
    )

print("Loaded the canonical 1.3.1 mobility panel for forecasting-sequence construction.")
display(round_display(path_status_df))
display(round_display(source_contract_df))
display(loaded_target_columns_df)

Loaded the canonical 1.3.1 mobility panel for forecasting-sequence construction.


,Asset,Path,Exists
0,analysis_ready_mobility_panel,pipeline_data/1.3.1.final_tables/analysis_ready_mobility_panel.parquet,True
1,3.3.1_support_panel,pipeline_data/3.3.1.final_tables/temporal_residual_mobility_panel.parquet,True


,Rows,Taxi Zones,First date,Last date,Dayparts,Temporal buckets,Forecast targets,Duplicate canonical rows,Missing daypart_order,Missing temporal_bucket_order,Missing observation_sequence_id
0,1559590,263,2023-01-01,2026-03-31,5,10,5,0,0,0,0


,Loaded forecast target columns
0,taxi_trip_count
1,taxi_avg_trip_speed
2,fhvhv_trip_count
3,fhvhv_avg_trip_speed
4,subway_ridership


Findings\. The canonical mobility panel contains 1,559,590 observations covering 263 Taxi Zones from January 1, 2023 through March 31, 2026, with all five dayparts, 10 temporal buckets, and the five selected forecasting targets present\. The source remains unique at the canonical grain with no duplicate rows, and the daypart\_order, temporal\_bucket\_order, and observation\_sequence\_id fields are complete\. Both required upstream assets are available, providing the canonical mobility observations and the 3\.3\.1 support provenance needed for the next stage\.

### 4.1.1.2 Construct and Validate Ordered Forecasting Sequences ###

This chapter converts the five forecasting targets into a common ordered sequence representation at the Taxi Zone × Metric level while preserving Date × Daypart as the observation context. The resulting structure follows the full within-day progression from Overnight through Evening and across successive dates, allowing future models to draw on both recent adjacent dayparts and longer historical patterns. Mechanical ordering checks and compact structural summaries verify the sequence grain, chronology, and target coverage before forecast horizons are introduced.

### Reshape the five metrics into ordered forecasting sequences

Reshape the five mobility targets from the wide canonical panel into one long forecasting table. A Taxi Zone × Metric pair identifies the time series, while Date, Daypart, Temporal Bucket, and the upstream sequence-ordering fields identify each successive observed mobility state.

In [3]:
# ---------------------------------------------------------------------
# 4.1.1.2A — Reshape the five metrics into ordered forecasting sequences
# ---------------------------------------------------------------------

sequence_identity_columns = [
    "taxi_zone_id",
    "zone",
    "borough",
    "canonical_location_id",
    "date",
    "daypart",
    "daypart_order",
    "temporal_bucket",
    "temporal_bucket_order",
    "observation_sequence_id",
    "pre_post_cp",
]

# Melt only the agreed forecasting targets so the long table becomes one
# canonical Taxi Zone × Metric sequence layer without dropping missing rows.
forecast_sequence_long_df = mobility_panel_df.melt(
    id_vars=sequence_identity_columns,
    value_vars=FORECAST_TARGET_METRICS,
    var_name="metric",
    value_name="observed_value",
)

forecast_sequence_long_df["forecast_sequence_id"] = (
    "zone="
    + forecast_sequence_long_df["taxi_zone_id"].astype(str)
    + " | metric="
    + forecast_sequence_long_df["metric"]
)

forecast_sequence_long_df = forecast_sequence_long_df.sort_values(
    ["taxi_zone_id", "metric", "observation_sequence_id"]
).reset_index(drop=True)

long_duplicate_rows = duplicate_count(
    forecast_sequence_long_df,
    LONG_SEQUENCE_GRAIN_COLUMNS,
)

wide_zero_count = int(
    mobility_panel_df[FORECAST_TARGET_METRICS]
    .fillna(np.nan)
    .eq(0)
    .sum()
    .sum()
)
long_zero_count = int(forecast_sequence_long_df["observed_value"].eq(0).sum())
wide_missing_count = int(mobility_panel_df[FORECAST_TARGET_METRICS].isna().sum().sum())
long_missing_count = int(forecast_sequence_long_df["observed_value"].isna().sum())

forecast_sequence_inventory_df = (
    forecast_sequence_long_df[
        ["taxi_zone_id", "zone", "borough", "metric", "forecast_sequence_id"]
    ]
    .drop_duplicates()
    .sort_values(["taxi_zone_id", "metric"])
    .reset_index(drop=True)
)

forecast_sequence_contract_df = pd.DataFrame(
    [
        {
            "Long-layer rows": len(forecast_sequence_long_df),
            "Taxi Zone × Metric sequences": forecast_sequence_inventory_df.shape[0],
            "Metrics represented": forecast_sequence_long_df["metric"].nunique(),
            "Duplicate sequence positions": long_duplicate_rows,
            "Missing sequence order": int(forecast_sequence_long_df["observation_sequence_id"].isna().sum()),
            "Minimum sequence position": forecast_sequence_long_df["observation_sequence_id"].min(),
            "Maximum sequence position": forecast_sequence_long_df["observation_sequence_id"].max(),
            "First date": forecast_sequence_long_df["date"].min(),
            "Last date": forecast_sequence_long_df["date"].max(),
            "Zero values preserved": wide_zero_count == long_zero_count,
            "Missing values preserved": wide_missing_count == long_missing_count,
        }
    ]
)

if RUN_FORECAST_SEQUENCE_QA:
    assert long_duplicate_rows == 0, (
        "The forecasting long layer is not unique at Taxi Zone × Metric × sequence position."
    )
    assert forecast_sequence_long_df["metric"].nunique() == len(FORECAST_TARGET_METRICS), (
        "The long forecasting layer does not contain all five agreed targets."
    )
    assert wide_zero_count == long_zero_count, "Observed zeros changed during the reshape."
    assert wide_missing_count == long_missing_count, "Missing observations changed during the reshape."

print("Constructed the canonical long forecasting-sequence layer.")
display(round_display(forecast_sequence_contract_df))

Constructed the canonical long forecasting-sequence layer.


,Long-layer rows,Taxi Zone × Metric sequences,Metrics represented,Duplicate sequence positions,Missing sequence order,Minimum sequence position,Maximum sequence position,First date,Last date,Zero values preserved,Missing values preserved
0,7797950,1315,5,0,0,1,5930,2023-01-01,2026-03-31,True,True


Findings\. Reshaping the five target metrics produces 7,797,950 long\-form observations across 1,315 Taxi Zone × Metric sequences, corresponding to 263 Taxi Zones for each of the five targets\. Sequence positions span 1 through 5,930 with no duplicate or missing sequence\-order values, while observed zeros and genuine missing values remain preserved\. The transformed layer therefore retains the full January 2023 through March 2026 study period without altering the underlying observation structure\.

### Validate daypart progression and chronological ordering

Validate that the transformed forecasting sequences preserve the established chronological progression across dayparts and dates. Because future forecast horizons will be defined as sequence-step offsets, the ordering contract must be both mechanically verified and visible in the notebook before target alignment is introduced.

In [4]:
# ---------------------------------------------------------------------
# 4.1.1.2B — Validate daypart progression and chronological ordering
# ---------------------------------------------------------------------

ordered_sequence_df = forecast_sequence_long_df.sort_values(
    ["taxi_zone_id", "metric", "observation_sequence_id"]
).copy()

# Compare each row to the next row within the same Taxi Zone × Metric sequence
# so the future horizon contract can rely on sequence-step offsets later on.
ordered_sequence_df["next_observation_sequence_id"] = (
    ordered_sequence_df.groupby(FORECAST_SEQUENCE_ID_COLUMNS)["observation_sequence_id"]
    .shift(-1)
)
ordered_sequence_df["next_daypart"] = (
    ordered_sequence_df.groupby(FORECAST_SEQUENCE_ID_COLUMNS)["daypart"]
    .shift(-1)
)
ordered_sequence_df["next_daypart_order"] = (
    ordered_sequence_df.groupby(FORECAST_SEQUENCE_ID_COLUMNS)["daypart_order"]
    .shift(-1)
)
ordered_sequence_df["next_date"] = (
    ordered_sequence_df.groupby(FORECAST_SEQUENCE_ID_COLUMNS)["date"]
    .shift(-1)
)
ordered_sequence_df["next_observed_value"] = (
    ordered_sequence_df.groupby(FORECAST_SEQUENCE_ID_COLUMNS)["observed_value"]
    .shift(-1)
)

transition_df = ordered_sequence_df.loc[
    ordered_sequence_df["next_observation_sequence_id"].notna()
].copy()

min_daypart_order = int(ordered_sequence_df["daypart_order"].min())
max_daypart_order = int(ordered_sequence_df["daypart_order"].max())

transition_df["sequence_step_gap"] = (
    transition_df["next_observation_sequence_id"]
    - transition_df["observation_sequence_id"]
)
transition_df["expected_next_daypart_order"] = np.where(
    transition_df["daypart_order"] < max_daypart_order,
    transition_df["daypart_order"] + 1,
    min_daypart_order,
)
transition_df["expected_next_date"] = np.where(
    transition_df["daypart_order"] < max_daypart_order,
    transition_df["date"],
    transition_df["date"] + pd.Timedelta(days=1),
)
transition_df["sequence_gap_is_expected"] = transition_df["sequence_step_gap"].eq(1)
transition_df["daypart_transition_is_expected"] = transition_df["next_daypart_order"].eq(
    transition_df["expected_next_daypart_order"]
)
transition_df["date_transition_is_expected"] = pd.to_datetime(
    transition_df["next_date"]
).eq(pd.to_datetime(transition_df["expected_next_date"]))
transition_df["transition_has_missing_observation"] = (
    transition_df["observed_value"].isna()
    | transition_df["next_observed_value"].isna()
)

sequence_order_qa_df = pd.DataFrame(
    {
        "Sequence-order QA": [
            "Checked consecutive transitions",
            "Expected sequence-step transitions",
            "Unexpected sequence-step transitions",
            "Unexpected daypart transitions",
            "Unexpected date transitions",
            "Transitions with missing observation on either side",
            "Duplicate sequence positions",
        ],
        "Count": [
            len(transition_df),
            int(transition_df["sequence_gap_is_expected"].sum()),
            int((~transition_df["sequence_gap_is_expected"]).sum()),
            int((~transition_df["daypart_transition_is_expected"]).sum()),
            int((~transition_df["date_transition_is_expected"]).sum()),
            int(transition_df["transition_has_missing_observation"].sum()),
            long_duplicate_rows,
        ],
    }
)

preview_anchor_df = (
    forecast_sequence_long_df.groupby(
        ["taxi_zone_id", "metric", "forecast_sequence_id"],
        as_index=False,
    )
    .agg(
        non_null_observed_rows=("observed_value", lambda series: int(series.notna().sum())),
        represented_dates=("date", "nunique"),
    )
    .sort_values(
        ["non_null_observed_rows", "represented_dates", "taxi_zone_id", "metric"],
        ascending=[False, False, True, True],
    )
)

preview_anchor = preview_anchor_df.iloc[0]
preview_series_df = forecast_sequence_long_df.loc[
    forecast_sequence_long_df["forecast_sequence_id"].eq(preview_anchor["forecast_sequence_id"]),
    [
        "taxi_zone_id",
        "metric",
        "observation_sequence_id",
        "date",
        "daypart",
        "temporal_bucket",
        "observed_value",
    ],
].sort_values("observation_sequence_id").reset_index(drop=True)

preview_series_df["next_date"] = preview_series_df["date"].shift(-1)
preview_transition_indices = preview_series_df.index[
    preview_series_df["date"].ne(preview_series_df["next_date"])
].tolist()

if preview_transition_indices:
    preview_anchor_index = int(preview_transition_indices[0])
    preview_start = max(0, preview_anchor_index - 5)
else:
    preview_start = 0

preview_end = min(len(preview_series_df), preview_start + 12)
preview_start = max(0, preview_end - 12)
ordered_sequence_preview_df = preview_series_df.iloc[
    preview_start:preview_end
].drop(columns=["next_date"])

if RUN_FORECAST_SEQUENCE_QA:
    assert int((~transition_df["sequence_gap_is_expected"]).sum()) == 0, (
        "The long forecasting layer contains unexpected sequence-step gaps."
    )
    assert int((~transition_df["daypart_transition_is_expected"]).sum()) == 0, (
        "The long forecasting layer contains unexpected daypart transitions."
    )
    assert int((~transition_df["date_transition_is_expected"]).sum()) == 0, (
        "The long forecasting layer contains unexpected date transitions."
    )

print("Validated sequence-step ordering and built a purposeful ordered preview.")
display(round_display(sequence_order_qa_df))
display(round_display(ordered_sequence_preview_df))

Validated sequence-step ordering and built a purposeful ordered preview.


,Sequence-order QA,Count
0,Checked consecutive transitions,7796635
1,Expected sequence-step transitions,7796635
2,Unexpected sequence-step transitions,0
3,Unexpected daypart transitions,0
4,Unexpected date transitions,0
5,Transitions with missing observation on either side,1325238
6,Duplicate sequence positions,0


,taxi_zone_id,metric,observation_sequence_id,date,daypart,temporal_bucket,observed_value
0,3,fhvhv_avg_trip_speed,1,2023-01-01,overnight,weekend_overnight,21.703
1,3,fhvhv_avg_trip_speed,2,2023-01-01,am_peak,weekend_am_peak,25.484
2,3,fhvhv_avg_trip_speed,3,2023-01-01,midday,weekend_midday,20.743
3,3,fhvhv_avg_trip_speed,4,2023-01-01,pm_peak,weekend_pm_peak,18.279
4,3,fhvhv_avg_trip_speed,5,2023-01-01,evening,weekend_evening,21.912
5,3,fhvhv_avg_trip_speed,6,2023-01-02,overnight,weekday_overnight,23.866
6,3,fhvhv_avg_trip_speed,7,2023-01-02,am_peak,weekday_am_peak,21.333
7,3,fhvhv_avg_trip_speed,8,2023-01-02,midday,weekday_midday,19.058
8,3,fhvhv_avg_trip_speed,9,2023-01-02,pm_peak,weekday_pm_peak,18.988
9,3,fhvhv_avg_trip_speed,10,2023-01-02,evening,weekday_evening,22.972


Findings\. All 7,796,635 consecutive sequence transitions follow the expected sequence\-step, daypart, and date progression, with no unexpected transitions or duplicate sequence positions\. The structural preview shows the intended five\-step daily cycle—Overnight, AM Peak, Midday, PM Peak, and Evening—followed by the next day's Overnight observation, while weekday/weekend temporal\-bucket labels remain aligned with their dates\. Approximately 1\.325 million transitions have a missing observed value on at least one side, but these missing measurements do not disrupt the underlying chronological sequence\.

### Summarize the canonical sequence layer

Summarize the resulting sequence structure by forecasting target so the observation coverage and time span are visible without printing the full long panel. The summary also confirms that each target uses one ordered sequence per Taxi Zone rather than separate daypart-specific forecasting series.

In [5]:
# ---------------------------------------------------------------------
# 4.1.1.2C — Summarize the canonical sequence layer
# ---------------------------------------------------------------------

forecast_sequence_metric_summary_df = (
    forecast_sequence_long_df.groupby("metric", as_index=False)
    .agg(
        taxi_zone_metric_sequence_count=("forecast_sequence_id", "nunique"),
        observation_row_count=("forecast_sequence_id", "size"),
        non_null_observation_count=("observed_value", lambda series: int(series.notna().sum())),
        first_date=("date", "min"),
        last_date=("date", "max"),
        represented_daypart_count=("daypart", "nunique"),
    )
)

forecast_sequence_metric_summary_df["non_null_observation_share"] = (
    forecast_sequence_metric_summary_df["non_null_observation_count"]
    / forecast_sequence_metric_summary_df["observation_row_count"]
)

print(
    "Canonical forecasting sequence layer shape: "
    f"rows={len(forecast_sequence_long_df):,}, "
    f"columns={forecast_sequence_long_df.shape[1]:,}, "
    f"unique Taxi Zone × Metric sequences={forecast_sequence_inventory_df.shape[0]:,}"
)

display(
    round_display(
        forecast_sequence_metric_summary_df[
            [
                "metric",
                "taxi_zone_metric_sequence_count",
                "observation_row_count",
                "non_null_observation_count",
                "non_null_observation_share",
                "first_date",
                "last_date",
                "represented_daypart_count",
            ]
        ]
    )
)

Canonical forecasting sequence layer shape: rows=7,797,950, columns=14, unique Taxi Zone × Metric sequences=1,315


,metric,taxi_zone_metric_sequence_count,observation_row_count,non_null_observation_count,non_null_observation_share,first_date,last_date,represented_daypart_count
0,fhvhv_avg_trip_speed,263,1559590,1523474,0.977,2023-01-01,2026-03-31,5
1,fhvhv_trip_count,263,1559590,1531505,0.982,2023-01-01,2026-03-31,5
2,subway_ridership,263,1559590,912213,0.585,2023-01-01,2026-03-31,5
3,taxi_avg_trip_speed,263,1559590,1125138,0.721,2023-01-01,2026-03-31,5
4,taxi_trip_count,263,1559590,1531505,0.982,2023-01-01,2026-03-31,5


Findings\. All five forecasting targets span the full study period, retain all five dayparts, and produce 263 Taxi Zone × Metric sequences each, but observed\-value coverage differs substantially by metric\. Taxi and FHVHV trip counts are the most complete at 98\.2%, followed by FHVHV average trip speed at 97\.7%; Taxi average trip speed has 72\.1% observed coverage, while Subway ridership is considerably sparser at 58\.5%\. These differences make sequence\-level support and continuity especially important for determining which zone–metric combinations can support forecasting\.

### Section Summary

The five selected mobility metrics have been organized into 1,315 chronologically ordered Taxi Zone × Metric sequences spanning 5,930 daypart steps each\. The sequence structure is mechanically consistent across dayparts and dates, with no ordering violations or duplicate positions, and it preserves missing observations rather than manufacturing continuity\. Coverage is strong for Taxi and FHVHV demand and FHVHV speed but materially lower for Taxi speed and Subway ridership, so the next chapter must distinguish complete sequence structure from sufficient observed support when determining forecasting eligibility\.

## 4.1.1.3 Assess Forecasting Sequence Support

Reuse the daypart-level support routing established in `3.3.1` and combine it with the continuity and coverage characteristics needed for forecasting. The prior modeled, fallback, and unavailable classifications describe suitability for the residualization workflow; they are retained as support provenance rather than treated automatically as forecasting eligibility.

### Load authoritative `3.3.1` support provenance

Load the finalized `3.3.1` handoff and retain only the fields needed to recover daypart-level series support for the five forecasting metrics. Residual values, fitted expected values, standardized residuals, and decomposition outputs remain outside the forecasting target layer.

In [6]:
# ---------------------------------------------------------------------
# 4.1.1.3A — Load authoritative 3.3.1 support provenance
# ---------------------------------------------------------------------

support_columns_to_load = [
    "taxi_zone_id",
    "metric",
    "daypart",
    "series_id",
    "support_status",
    "support_reason",
    "eligibility_pathway",
]

support_source_df = pd.read_parquet(
    RESIDUAL_SUPPORT_PANEL_PATH,
    columns=support_columns_to_load,
)

# The 3.3.1 residual panel repeats the support fields across dates, so we
# validate constancy first and then reduce to one Taxi Zone × Metric × Daypart row.
require_columns(
    support_source_df,
    support_columns_to_load,
    "support_source_df",
)

support_source_df = support_source_df.loc[
    support_source_df["metric"].isin(FORECAST_TARGET_METRICS)
].copy()

support_constancy_df = (
    support_source_df.groupby(DAYPART_SUPPORT_GRAIN_COLUMNS, as_index=False)
    .agg(
        distinct_series_id_count=("series_id", "nunique"),
        distinct_support_status_count=("support_status", "nunique"),
        distinct_support_reason_count=("support_reason", "nunique"),
        distinct_eligibility_pathway_count=("eligibility_pathway", "nunique"),
    )
)

support_provenance_df = (
    support_source_df.groupby(DAYPART_SUPPORT_GRAIN_COLUMNS, as_index=False)
    .agg(
        support_series_id=("series_id", "first"),
        support_status=("support_status", "first"),
        support_reason=("support_reason", "first"),
        eligibility_pathway=("eligibility_pathway", "first"),
    )
    .sort_values(["taxi_zone_id", "metric", "daypart"])
    .reset_index(drop=True)
)

support_status_values_present = sorted(
    support_provenance_df["support_status"].dropna().astype(str).unique().tolist()
)

support_source_contract_df = pd.DataFrame(
    [
        {
            "Support rows": len(support_provenance_df),
            "Unique Taxi Zones": support_provenance_df["taxi_zone_id"].nunique(),
            "Metrics": support_provenance_df["metric"].nunique(),
            "Dayparts": support_provenance_df["daypart"].nunique(),
            "Duplicate Taxi Zone × Metric × Daypart records": duplicate_count(
                support_provenance_df,
                DAYPART_SUPPORT_GRAIN_COLUMNS,
            ),
            "Support-status values present": ", ".join(support_status_values_present),
        }
    ]
)

retained_support_fields_df = pd.DataFrame(
    {
        "Retained support fields": [
            "taxi_zone_id",
            "metric",
            "daypart",
            "support_series_id",
            "support_status",
            "support_reason",
            "eligibility_pathway",
        ]
    }
)

if RUN_FORECAST_SEQUENCE_QA:
    assert (
        support_constancy_df[
            [
                "distinct_series_id_count",
                "distinct_support_status_count",
                "distinct_support_reason_count",
                "distinct_eligibility_pathway_count",
            ]
        ]
        .le(1)
        .all()
        .all()
    ), "3.3.1 support provenance is not constant within Taxi Zone × Metric × Daypart series."
    assert duplicate_count(support_provenance_df, DAYPART_SUPPORT_GRAIN_COLUMNS) == 0, (
        "Reduced 3.3.1 support provenance is not unique at Taxi Zone × Metric × Daypart grain."
    )

print("Reduced the 3.3.1 residual handoff to daypart-level support provenance for the five forecasting targets.")
display(round_display(support_source_contract_df))
display(retained_support_fields_df)

Reduced the 3.3.1 residual handoff to daypart-level support provenance for the five forecasting targets.


,Support rows,Unique Taxi Zones,Metrics,Dayparts,Duplicate Taxi Zone × Metric × Daypart records,Support-status values present
0,6575,263,5,5,0,"fallback, modeled, unavailable"


,Retained support fields
0,taxi_zone_id
1,metric
2,daypart
3,support_series_id
4,support_status
5,support_reason
6,eligibility_pathway


Findings\. The 3\.3\.1 support handoff provides complete daypart\-level provenance for the forecasting population: 6,575 unique Taxi Zone × Metric × Daypart records, representing all 263 Taxi Zones, five forecasting metrics, and five dayparts\. No duplicate support records are present, and the retained classifications span the expected modeled, fallback, and unavailable statuses\. The reduced layer therefore provides a complete and uniquely keyed record of the upstream support decision for every daypart contributing to the unified forecasting sequences\.

### Reconcile daypart support with unified forecasting sequences

Summarize the five daypart-level support classifications within each Taxi Zone × Metric forecasting sequence. This identifies sequences with uniformly modeled dayparts, mixtures of modeled and fallback dayparts, and structural absence in one or more parts of the daily cycle without yet assigning forecasting eligibility.

In [7]:
# ---------------------------------------------------------------------
# 4.1.1.3B — Reconcile daypart support with unified forecasting sequences
# ---------------------------------------------------------------------

forecast_sequence_support_summary_df = (
    support_provenance_df.groupby(FORECAST_SEQUENCE_ID_COLUMNS, as_index=False)
    .agg(
        modeled_dayparts=("support_status", lambda series: int(series.eq("modeled").sum())),
        fallback_dayparts=("support_status", lambda series: int(series.eq("fallback").sum())),
        unavailable_dayparts=("support_status", lambda series: int(series.eq("unavailable").sum())),
        represented_support_dayparts=("daypart", "nunique"),
    )
)

forecast_sequence_support_summary_df["support_pattern"] = (
    forecast_sequence_support_summary_df.apply(
        lambda row: classify_support_pattern(
            modeled_dayparts=int(row["modeled_dayparts"]),
            fallback_dayparts=int(row["fallback_dayparts"]),
            unavailable_dayparts=int(row["unavailable_dayparts"]),
            represented_dayparts=int(row["represented_support_dayparts"]),
        ),
        axis=1,
    )
)

forecast_sequence_with_support_df = forecast_sequence_inventory_df.merge(
    forecast_sequence_support_summary_df,
    on=FORECAST_SEQUENCE_ID_COLUMNS,
    how="left",
    validate="one_to_one",
)

forecast_sequence_with_support_df["support_pattern"] = (
    forecast_sequence_with_support_df["support_pattern"]
    .fillna("support provenance missing")
)

for support_count_column in [
    "modeled_dayparts",
    "fallback_dayparts",
    "unavailable_dayparts",
    "represented_support_dayparts",
]:
    forecast_sequence_with_support_df[support_count_column] = (
        forecast_sequence_with_support_df[support_count_column]
        .fillna(0)
        .astype(int)
    )

support_merge_qa_df = pd.DataFrame(
    [
        {
            "Forecasting sequences": len(forecast_sequence_inventory_df),
            "Sequences matched to 3.3.1 support": int(
                forecast_sequence_with_support_df["represented_support_dayparts"].gt(0).sum()
            ),
            "Unmatched sequences": int(
                forecast_sequence_with_support_df["represented_support_dayparts"].eq(0).sum()
            ),
            "Duplicate support summaries": duplicate_count(
                forecast_sequence_support_summary_df,
                FORECAST_SEQUENCE_ID_COLUMNS,
            ),
            "Sequences with five represented support dayparts": int(
                forecast_sequence_with_support_df["represented_support_dayparts"].eq(5).sum()
            ),
        }
    ]
)

support_pattern_preview_df = (
    forecast_sequence_with_support_df.sort_values(
        [
            "support_pattern",
            "taxi_zone_id",
            "metric",
        ]
    )
    .drop_duplicates(subset=["support_pattern"])
    .loc[
        :,
        [
            "taxi_zone_id",
            "metric",
            "modeled_dayparts",
            "fallback_dayparts",
            "unavailable_dayparts",
            "support_pattern",
        ],
    ]
    .reset_index(drop=True)
)

if RUN_FORECAST_SEQUENCE_QA:
    assert duplicate_count(
        forecast_sequence_support_summary_df,
        FORECAST_SEQUENCE_ID_COLUMNS,
    ) == 0, "Support summaries are not unique at Taxi Zone × Metric grain."

print("Summarized daypart support across unified Taxi Zone × Metric forecasting sequences.")
display(round_display(support_merge_qa_df))
display(round_display(support_pattern_preview_df))

Summarized daypart support across unified Taxi Zone × Metric forecasting sequences.


,Forecasting sequences,Sequences matched to 3.3.1 support,Unmatched sequences,Duplicate support summaries,Sequences with five represented support dayparts
0,1315,1315,0,0,1315


,taxi_zone_id,metric,modeled_dayparts,fallback_dayparts,unavailable_dayparts,support_pattern
0,1,fhvhv_avg_trip_speed,0,5,0,all fallback
1,4,fhvhv_avg_trip_speed,5,0,0,all modeled
2,1,subway_ridership,0,0,5,all unavailable
3,110,taxi_avg_trip_speed,0,2,3,contains unavailable
4,3,fhvhv_avg_trip_speed,4,1,0,modeled + fallback


Findings\. All 1,315 Taxi Zone × Metric forecasting sequences match successfully to 3\.3\.1 support provenance, with no unmatched sequences or duplicate support summaries, and every sequence has support information for all five dayparts\. The examples also confirm that the unified sequences capture several materially different support patterns, ranging from five modeled dayparts to mixed modeled/fallback cases, all\-fallback sequences, and structural unavailability\. This preserves the upstream support evidence while allowing forecasting suitability to be assessed at the broader Taxi Zone × Metric sequence level\.

### Compute forecasting-specific continuity and coverage diagnostics

Measure only the additional sequence properties needed for forecasting that were not resolved by the daypart-level residual support routing. Focus on observed-history depth, continuity through the ordered daypart sequence, and coverage of the post-congestion-pricing development period and proposed 2026 evaluation period.

In [8]:
# ---------------------------------------------------------------------
# 4.1.1.3C — Compute forecasting-specific continuity and coverage diagnostics
# ---------------------------------------------------------------------

sequence_long_columns = [
    "taxi_zone_id",
    "zone",
    "borough",
    "metric",
    "forecast_sequence_id",
    "date",
    "daypart",
    "daypart_order",
    "temporal_bucket",
    "temporal_bucket_order",
    "observation_sequence_id",
    "pre_post_cp",
    "observed_value",
]

sequence_long_for_diagnostics_df = forecast_sequence_long_df.loc[
    :,
    sequence_long_columns
].copy()


def summarize_forecasting_sequence(sequence_df: pd.DataFrame) -> pd.Series:
    """Summarize one Taxi Zone × Metric sequence without dropping missing rows first."""
    sequence_df = sequence_df.sort_values("observation_sequence_id").reset_index(drop=True)
    observed_mask = sequence_df["observed_value"].notna().to_numpy()
    observed_positions = np.flatnonzero(observed_mask)

    total_sequence_rows = int(len(sequence_df))
    non_null_observed_rows = int(observed_mask.sum())
    overall_coverage_rate = (
        non_null_observed_rows / total_sequence_rows
        if total_sequence_rows
        else np.nan
    )

    pre_cp_mask = sequence_df["pre_post_cp"].eq("pre_cp").to_numpy()
    post_cp_mask = sequence_df["pre_post_cp"].eq("post_cp").to_numpy()
    development_mask = sequence_df["date"].between(
        FORECAST_DEVELOPMENT_START_DATE,
        FORECAST_DEVELOPMENT_END_DATE,
    ).to_numpy()
    evaluation_mask = sequence_df["date"].between(
        FORECAST_EVALUATION_START_DATE,
        FORECAST_EVALUATION_END_DATE,
    ).to_numpy()

    if len(observed_positions):
        first_idx = int(observed_positions[0])
        last_idx = int(observed_positions[-1])
        first_non_null_sequence_position = int(sequence_df.loc[first_idx, "observation_sequence_id"])
        last_non_null_sequence_position = int(sequence_df.loc[last_idx, "observation_sequence_id"])
        first_non_null_timestamp = pd.Timestamp(sequence_df.loc[first_idx, "date"])
        last_non_null_timestamp = pd.Timestamp(sequence_df.loc[last_idx, "date"])
        active_sequence_span = (
            last_non_null_sequence_position
            - first_non_null_sequence_position
            + 1
        )
    else:
        first_non_null_sequence_position = np.nan
        last_non_null_sequence_position = np.nan
        first_non_null_timestamp = pd.NaT
        last_non_null_timestamp = pd.NaT
        active_sequence_span = 0

    pre_cp_possible_rows = int(pre_cp_mask.sum())
    post_cp_possible_rows = int(post_cp_mask.sum())
    development_possible_rows = int(development_mask.sum())
    evaluation_possible_rows = int(evaluation_mask.sum())

    pre_cp_observed_rows = int(np.logical_and(observed_mask, pre_cp_mask).sum())
    post_cp_observed_rows = int(np.logical_and(observed_mask, post_cp_mask).sum())
    development_observed_rows = int(np.logical_and(observed_mask, development_mask).sum())
    evaluation_observed_rows = int(np.logical_and(observed_mask, evaluation_mask).sum())

    return pd.Series(
        {
            "zone": sequence_df.loc[0, "zone"],
            "borough": sequence_df.loc[0, "borough"],
            "total_sequence_rows": total_sequence_rows,
            "non_null_observed_rows": non_null_observed_rows,
            "overall_coverage_rate": overall_coverage_rate,
            "first_non_null_sequence_position": first_non_null_sequence_position,
            "last_non_null_sequence_position": last_non_null_sequence_position,
            "first_non_null_timestamp": first_non_null_timestamp,
            "last_non_null_timestamp": last_non_null_timestamp,
            "active_sequence_span": int(active_sequence_span),
            "longest_consecutive_observed_run": longest_true_run(observed_mask),
            "missing_run_count": count_false_runs(observed_mask),
            "max_missing_run_length": max_false_run_length(observed_mask),
            "pre_cp_possible_rows": pre_cp_possible_rows,
            "pre_cp_observed_rows": pre_cp_observed_rows,
            "pre_cp_observed_share": (
                pre_cp_observed_rows / pre_cp_possible_rows
                if pre_cp_possible_rows
                else np.nan
            ),
            "post_cp_possible_rows": post_cp_possible_rows,
            "post_cp_observed_rows": post_cp_observed_rows,
            "post_cp_observed_share": (
                post_cp_observed_rows / post_cp_possible_rows
                if post_cp_possible_rows
                else np.nan
            ),
            "development_possible_rows": development_possible_rows,
            "development_observed_rows": development_observed_rows,
            "development_observed_share": (
                development_observed_rows / development_possible_rows
                if development_possible_rows
                else np.nan
            ),
            "evaluation_possible_rows": evaluation_possible_rows,
            "evaluation_observed_rows": evaluation_observed_rows,
            "evaluation_observed_share": (
                evaluation_observed_rows / evaluation_possible_rows
                if evaluation_possible_rows
                else np.nan
            ),
        }
    )


if REBUILD_FORECAST_SEQUENCE_SUPPORT_DIAGNOSTICS or not FORECAST_SEQUENCE_SUPPORT_DIAGNOSTICS_PATH.exists():
    print("Computing Taxi Zone × Metric continuity diagnostics...", flush=True)

    diagnostic_rows = []
    grouped_sequences = sequence_long_for_diagnostics_df.groupby(
        FORECAST_SEQUENCE_ID_COLUMNS + ["forecast_sequence_id"],
        dropna=False,
        sort=True,
    )

    print(f"Sequences to summarize: {grouped_sequences.ngroups:,}", flush=True)

    for (
        taxi_zone_id,
        metric,
        forecast_sequence_id,
    ), sequence_group_df in grouped_sequences:
        sequence_summary = summarize_forecasting_sequence(sequence_group_df)
        sequence_summary["taxi_zone_id"] = taxi_zone_id
        sequence_summary["metric"] = metric
        sequence_summary["forecast_sequence_id"] = forecast_sequence_id
        diagnostic_rows.append(sequence_summary)

    sequence_diagnostics_df = pd.DataFrame(diagnostic_rows)
    sequence_diagnostics_df = sequence_diagnostics_df[
        [
            "taxi_zone_id",
            "zone",
            "borough",
            "metric",
            "forecast_sequence_id",
            "total_sequence_rows",
            "non_null_observed_rows",
            "overall_coverage_rate",
            "first_non_null_sequence_position",
            "last_non_null_sequence_position",
            "first_non_null_timestamp",
            "last_non_null_timestamp",
            "active_sequence_span",
            "longest_consecutive_observed_run",
            "missing_run_count",
            "max_missing_run_length",
            "pre_cp_possible_rows",
            "pre_cp_observed_rows",
            "pre_cp_observed_share",
            "post_cp_possible_rows",
            "post_cp_observed_rows",
            "post_cp_observed_share",
            "development_possible_rows",
            "development_observed_rows",
            "development_observed_share",
            "evaluation_possible_rows",
            "evaluation_observed_rows",
            "evaluation_observed_share",
        ]
    ]

    sequence_diagnostics_df = sequence_diagnostics_df.merge(
        forecast_sequence_with_support_df[
            [
                "taxi_zone_id",
                "metric",
                "modeled_dayparts",
                "fallback_dayparts",
                "unavailable_dayparts",
                "represented_support_dayparts",
                "support_pattern",
            ]
        ],
        on=FORECAST_SEQUENCE_ID_COLUMNS,
        how="left",
        validate="one_to_one",
    )

    sequence_diagnostics_df.to_parquet(
        FORECAST_SEQUENCE_SUPPORT_DIAGNOSTICS_PATH,
        index=False,
    )
    print(f"Wrote cached diagnostics: {FORECAST_SEQUENCE_SUPPORT_DIAGNOSTICS_PATH}", flush=True)
else:
    print(f"Loading cached diagnostics: {FORECAST_SEQUENCE_SUPPORT_DIAGNOSTICS_PATH}", flush=True)
    sequence_diagnostics_df = pd.read_parquet(
        FORECAST_SEQUENCE_SUPPORT_DIAGNOSTICS_PATH
    )

for timestamp_column in [
    "first_non_null_timestamp",
    "last_non_null_timestamp",
]:
    if timestamp_column in sequence_diagnostics_df.columns:
        sequence_diagnostics_df[timestamp_column] = pd.to_datetime(
            sequence_diagnostics_df[timestamp_column]
        )

coverage_columns = [
    "overall_coverage_rate",
    "pre_cp_observed_share",
    "post_cp_observed_share",
    "development_observed_share",
    "evaluation_observed_share",
]

invalid_coverage_rate_count = int(
    (~sequence_diagnostics_df[coverage_columns].stack(dropna=True).between(0, 1)).sum()
)

impossible_count_or_run_count = int(
    (
        sequence_diagnostics_df["non_null_observed_rows"].gt(sequence_diagnostics_df["total_sequence_rows"])
        | sequence_diagnostics_df["longest_consecutive_observed_run"].gt(sequence_diagnostics_df["total_sequence_rows"])
        | sequence_diagnostics_df["pre_cp_observed_rows"].gt(sequence_diagnostics_df["pre_cp_possible_rows"])
        | sequence_diagnostics_df["post_cp_observed_rows"].gt(sequence_diagnostics_df["post_cp_possible_rows"])
        | sequence_diagnostics_df["development_observed_rows"].gt(sequence_diagnostics_df["development_possible_rows"])
        | sequence_diagnostics_df["evaluation_observed_rows"].gt(sequence_diagnostics_df["evaluation_possible_rows"])
    ).sum()
)

unmatched_support_provenance_count = int(
    sequence_diagnostics_df["support_pattern"].eq("support provenance missing").sum()
)

sequence_diagnostic_contract_df = pd.DataFrame(
    [
        {
            "Diagnostic rows": len(sequence_diagnostics_df),
            "Unique Taxi Zone × Metric sequences": sequence_diagnostics_df[
                FORECAST_SEQUENCE_ID_COLUMNS
            ].drop_duplicates().shape[0],
            "Duplicates": duplicate_count(sequence_diagnostics_df, FORECAST_SEQUENCE_ID_COLUMNS),
            "Invalid coverage rates": invalid_coverage_rate_count,
            "Impossible count/run values": impossible_count_or_run_count,
            "Unmatched support provenance": unmatched_support_provenance_count,
        }
    ]
)

diagnostic_column_groups_df = compact_column_groups(
    {
        "Identity": [
            "taxi_zone_id",
            "zone",
            "borough",
            "metric",
            "forecast_sequence_id",
        ],
        "Coverage": [
            "total_sequence_rows",
            "non_null_observed_rows",
            "overall_coverage_rate",
            "longest_consecutive_observed_run",
            "missing_run_count",
            "max_missing_run_length",
        ],
        "Temporal extent": [
            "first_non_null_sequence_position",
            "last_non_null_sequence_position",
            "first_non_null_timestamp",
            "last_non_null_timestamp",
            "active_sequence_span",
        ],
        "Period coverage": [
            "pre_cp_observed_share",
            "post_cp_observed_share",
            "development_observed_share",
            "evaluation_observed_share",
        ],
        "Inherited support": [
            "modeled_dayparts",
            "fallback_dayparts",
            "unavailable_dayparts",
            "represented_support_dayparts",
            "support_pattern",
        ],
    }
)

if RUN_FORECAST_SEQUENCE_QA:
    assert duplicate_count(sequence_diagnostics_df, FORECAST_SEQUENCE_ID_COLUMNS) == 0, (
        "Forecasting diagnostics are not unique at Taxi Zone × Metric grain."
    )
    assert invalid_coverage_rate_count == 0, "Coverage rates fall outside [0, 1]."
    assert impossible_count_or_run_count == 0, "At least one diagnostic row has impossible counts or run lengths."

print("Built the forecasting-sequence continuity diagnostics and validated their contract.")
display(round_display(sequence_diagnostic_contract_df))
display(diagnostic_column_groups_df)

Loading cached diagnostics: pipeline_data/4.1.1.intermediate_tables/forecast_sequence_support_diagnostics.parquet
Built the forecasting-sequence continuity diagnostics and validated their contract.


,Diagnostic rows,Unique Taxi Zone × Metric sequences,Duplicates,Invalid coverage rates,Impossible count/run values,Unmatched support provenance
0,1315,1315,0,0,0,0


,Column group,Columns
0,Identity,"taxi_zone_id, zone, borough, metric, forecast_sequence_id"
1,Coverage,"total_sequence_rows, non_null_observed_rows, overall_coverage_rate, longest_consecutive_observed_run, missing_run_co..."
2,Temporal extent,"first_non_null_sequence_position, last_non_null_sequence_position, first_non_null_timestamp, last_non_null_timestamp..."
3,Period coverage,"pre_cp_observed_share, post_cp_observed_share, development_observed_share, evaluation_observed_share"
4,Inherited support,"modeled_dayparts, fallback_dayparts, unavailable_dayparts, represented_support_dayparts, support_pattern"


Findings\. The forecasting\-support diagnostic layer contains exactly one valid record for each of the 1,315 Taxi Zone × Metric sequences, with no duplicate records, invalid coverage rates, impossible count or run values, or unmatched upstream support\. The resulting diagnostics combine overall observation coverage, consecutive\-history depth, temporal extent, pre/post\-CP and development/evaluation coverage, and inherited daypart support in a single sequence\-level view\. This provides the necessary evidence for evaluating forecasting support without altering or filling gaps in the underlying mobility observations\.

### Build decision summaries

Summarize the inherited support patterns and forecasting-specific continuity measures at the target level and by major support pattern. These tables provide the evidence needed to decide how mixed fallback or unavailable dayparts should affect forecasting eligibility without requiring row-by-row inspection of every Taxi Zone.

In [9]:
# ---------------------------------------------------------------------
# 4.1.1.3D — Build decision summaries
# ---------------------------------------------------------------------

forecast_target_support_summary_df = (
    sequence_diagnostics_df.groupby("metric", as_index=False)
    .agg(
        taxi_zone_metric_sequence_count=("forecast_sequence_id", "nunique"),
        all_dayparts_modeled_count=(
            "support_pattern",
            lambda series: int(series.eq("all modeled").sum()),
        ),
        contains_fallback_dayparts_count=(
            "fallback_dayparts",
            lambda series: int(series.gt(0).sum()),
        ),
        contains_unavailable_dayparts_count=(
            "unavailable_dayparts",
            lambda series: int(series.gt(0).sum()),
        ),
        median_overall_coverage_rate=("overall_coverage_rate", "median"),
        median_longest_consecutive_observed_run=("longest_consecutive_observed_run", "median"),
        median_development_observed_share=("development_observed_share", "median"),
        median_evaluation_observed_share=("evaluation_observed_share", "median"),
    )
)

forecast_target_support_summary_df["all_dayparts_modeled_share"] = (
    forecast_target_support_summary_df["all_dayparts_modeled_count"]
    / forecast_target_support_summary_df["taxi_zone_metric_sequence_count"]
)
forecast_target_support_summary_df["contains_fallback_dayparts_share"] = (
    forecast_target_support_summary_df["contains_fallback_dayparts_count"]
    / forecast_target_support_summary_df["taxi_zone_metric_sequence_count"]
)
forecast_target_support_summary_df["contains_unavailable_dayparts_share"] = (
    forecast_target_support_summary_df["contains_unavailable_dayparts_count"]
    / forecast_target_support_summary_df["taxi_zone_metric_sequence_count"]
)

support_pattern_characteristics_df = (
    sequence_diagnostics_df.groupby("support_pattern", as_index=False)
    .agg(
        sequence_count=("forecast_sequence_id", "nunique"),
        metric_mix=(
            "metric",
            lambda series: ", ".join(sorted(pd.Series(series).dropna().unique())),
        ),
        median_overall_coverage_rate=("overall_coverage_rate", "median"),
        median_longest_consecutive_observed_run=("longest_consecutive_observed_run", "median"),
        median_development_observed_share=("development_observed_share", "median"),
        median_evaluation_observed_share=("evaluation_observed_share", "median"),
    )
    .sort_values(["sequence_count", "support_pattern"], ascending=[False, True])
    .reset_index(drop=True)
)

print(
    "Decision summaries represent "
    f"{sequence_diagnostics_df['forecast_sequence_id'].nunique():,} Taxi Zone × Metric sequences "
    f"from {len(sequence_diagnostics_df):,} diagnostic rows."
)

display(
    round_display(
        forecast_target_support_summary_df[
            [
                "metric",
                "taxi_zone_metric_sequence_count",
                "all_dayparts_modeled_count",
                "all_dayparts_modeled_share",
                "contains_fallback_dayparts_count",
                "contains_fallback_dayparts_share",
                "contains_unavailable_dayparts_count",
                "contains_unavailable_dayparts_share",
                "median_overall_coverage_rate",
                "median_longest_consecutive_observed_run",
                "median_development_observed_share",
                "median_evaluation_observed_share",
            ]
        ]
    )
)

display(round_display(support_pattern_characteristics_df))

Decision summaries represent 1,315 Taxi Zone × Metric sequences from 1,315 diagnostic rows.


,metric,taxi_zone_metric_sequence_count,all_dayparts_modeled_count,all_dayparts_modeled_share,contains_fallback_dayparts_count,contains_fallback_dayparts_share,contains_unavailable_dayparts_count,contains_unavailable_dayparts_share,median_overall_coverage_rate,median_longest_consecutive_observed_run,median_development_observed_share,median_evaluation_observed_share
0,fhvhv_avg_trip_speed,263,82,0.312,181,0.688,1,0.004,1.000,5930.0,1.000,1.000
1,fhvhv_trip_count,263,172,0.654,91,0.346,0,0.000,1.000,5930.0,1.000,1.000
2,subway_ridership,263,46,0.175,110,0.418,107,0.407,0.995,2738.0,0.997,0.981
3,taxi_avg_trip_speed,263,61,0.232,202,0.768,1,0.004,0.801,276.0,0.970,0.977
4,taxi_trip_count,263,169,0.643,94,0.357,0,0.000,1.000,5930.0,1.000,1.000


,support_pattern,sequence_count,metric_mix,median_overall_coverage_rate,median_longest_consecutive_observed_run,median_development_observed_share,median_evaluation_observed_share
0,all modeled,530,"fhvhv_avg_trip_speed, fhvhv_trip_count, subway_ridership, taxi_avg_trip_speed, taxi_trip_count",1.000,5930.0,1.000,1.000
1,modeled + fallback,472,"fhvhv_avg_trip_speed, fhvhv_trip_count, subway_ridership, taxi_avg_trip_speed, taxi_trip_count",1.000,5930.0,1.000,1.000
2,all fallback,204,"fhvhv_avg_trip_speed, fhvhv_trip_count, subway_ridership, taxi_avg_trip_speed, taxi_trip_count",0.993,1934.0,0.995,0.981
3,all unavailable,107,subway_ridership,0.000,0.0,0.000,0.000
4,contains unavailable,2,"fhvhv_avg_trip_speed, taxi_avg_trip_speed",0.001,1.5,0.001,0.000


Findings\. Support varies substantially by metric, but the upstream fallback designation does not by itself indicate poor forecasting history\. Taxi and FHVHV trip counts have complete median coverage and full 5,930\-step median runs, while FHVHV speed also has complete median coverage despite 68\.8% of its sequences containing at least one fallback daypart\. Taxi speed is more fragmented, with 80\.1% median overall coverage and a median longest uninterrupted run of 276 steps, although median coverage rises to 97\.0% during the 2025 development period and 97\.7% during the proposed 2026 evaluation period\. Subway has the largest structural limitation: 107 of 263 sequences are entirely unavailable, while the remaining population retains high median observed coverage\. Across all metrics, 530 sequences are fully modeled, 472 combine modeled and fallback dayparts, and another 204 are entirely fallback but still show 99\.3% median coverage; only 109 sequences contain structural unavailability, 107 of them Subway\. These patterns support treating forecasting eligibility separately from the earlier residual\-model routing rather than automatically excluding fallback sequences\.

Focus the remaining support review on Taxi Zone × Metric sequences with no structurally unavailable dayparts\. Summarize the lower tail of development\-period coverage, evaluation\-period coverage, and uninterrupted observed\-history length by metric so the forecasting eligibility threshold can be based on the actual distribution of usable history rather than the earlier residual\-model routing\.

In [10]:
# ---------------------------------------------------------------------
# 4.1.1.3E — Focus the remaining support review on Taxi Zone × Metric 
#            sequences with no structurally unavailable dayparts
# ---------------------------------------------------------------------

# ---------------------------------------------------------------------------
# Review the lower tail of forecasting support among structurally available
# Taxi Zone × Metric sequences. The purpose is to identify whether a small
# number of thin series separate naturally from the otherwise well-supported
# forecasting population.
# ---------------------------------------------------------------------------

diagnostics_path = Path(
    "pipeline_data/4.1.1.intermediate_tables/"
    "forecast_sequence_support_diagnostics.parquet"
)

# Reuse the in-memory diagnostics when available; otherwise load the cached
# sequence-level table produced in 4.1.1.3C.
if "forecast_sequence_support_diagnostics" in globals():
    support_diag = forecast_sequence_support_diagnostics.copy()
else:
    support_diag = pd.read_parquet(diagnostics_path)

# Structural unavailability is handled separately: a zone cannot support a
# forecast for a metric that is not observationally available there.
eligible_for_support_review = support_diag.loc[
    support_diag["unavailable_dayparts"].eq(0)
].copy()

# Review only the diagnostics that directly inform the minimum-history rule.
support_measures = {
    "development_observed_share": "Development observed share",
    "evaluation_observed_share": "Evaluation observed share",
    "longest_consecutive_observed_run": "Longest consecutive observed run",
}

summary_rows = []

for metric, metric_df in eligible_for_support_review.groupby("metric", sort=True):
    for column, label in support_measures.items():
        values = metric_df[column].dropna()

        summary_rows.append(
            {
                "metric": metric,
                "diagnostic": label,
                "sequence_count": len(values),
                "minimum": values.min(),
                "p05": values.quantile(0.05),
                "p25": values.quantile(0.25),
                "median": values.median(),
            }
        )

forecast_support_lower_tail = pd.DataFrame(summary_rows)

# Keep displayed precision compact while leaving the underlying dataframe
# unchanged for subsequent eligibility-rule construction.
display(
    forecast_support_lower_tail.style.format(
        {
            "minimum": "{:.3f}",
            "p05": "{:.3f}",
            "p25": "{:.3f}",
            "median": "{:.3f}",
        }
    )
)

,metric,diagnostic,sequence_count,minimum,p05,p25,median
0,fhvhv_avg_trip_speed,Development observed share,262,0.007,0.989,1.000,1.000
1,fhvhv_avg_trip_speed,Evaluation observed share,262,0.007,0.991,1.000,1.000
2,fhvhv_avg_trip_speed,Longest consecutive observed run,262,1.000,564.850,5930.000,5930.000
3,fhvhv_trip_count,Development observed share,263,0.026,0.993,1.000,1.000
4,fhvhv_trip_count,Evaluation observed share,263,0.019,0.991,1.000,1.000
5,fhvhv_trip_count,Longest consecutive observed run,263,2.000,1044.500,5930.000,5930.000
6,subway_ridership,Development observed share,156,0.000,0.984,1.000,1.000
7,subway_ridership,Evaluation observed share,156,0.000,0.974,0.981,0.981
8,subway_ridership,Longest consecutive observed run,156,75.000,1472.000,3798.250,5922.000
9,taxi_avg_trip_speed,Development observed share,262,0.003,0.084,0.825,0.970


Findings\. Among sequences without structural unavailability, Taxi and FHVHV trip counts are exceptionally well supported: their 5th\-percentile development and evaluation coverage remains above 99%, with 5th\-percentile uninterrupted runs exceeding 1,000 sequence steps\. FHVHV speed shows a similarly strong distribution, with approximately 99% coverage at the 5th percentile and long observed runs despite a very small extreme lower tail\. The 156 available Subway sequences are also generally well supported, with 5th\-percentile coverage of 98\.4% during development and 97\.4% during evaluation and substantial uninterrupted histories, although a few isolated sequences remain very sparse\. Taxi average trip speed is the clear exception: its 5th\-percentile coverage falls to roughly 8%, its 25th\-percentile coverage is only 82\.5–84\.7%, and the lower quartile contains much shorter uninterrupted runs\. Forecasting\-support exclusions can therefore be highly selective for Taxi trips, FHVHV trips, FHVHV speed, and available Subway series, while Taxi speed will require a more consequential eligibility rule because limited history affects a meaningful portion of its sequence population rather than only a handful of outliers\.

Apply the 80% post\-CP support rule\. Mark each Taxi Zone × Metric sequence as structurally available or unavailable, then flag whether it has at least 80% observed coverage in the post\-CP development period\. This creates the support fields that later forecasting notebooks can use consistently; pre\-CP and full\-history eligibility will be calculated separately once those training windows are defined in 4\.1\.2\.

In [11]:
# 4.1.1.3F — Apply the 80% post-CP support rule. 
#
# Eligibility must ultimately be evaluated against the actual training history
# available to each modeling track. This block therefore records:
#   1) regime-independent structural availability, and
#   2) support for the already-defined post-CP development period.
#
# The 2026 evaluation period is intentionally excluded from this eligibility
# decision so held-out observations do not determine whether a series qualifies
# for training.

MIN_FORECAST_TRAINING_COVERAGE = 0.80

diagnostics_path = Path(
    "pipeline_data/4.1.1.intermediate_tables/"
    "forecast_sequence_support_diagnostics.parquet"
)

# Reuse the sequence-level diagnostics already loaded in 4.1.1.3E when
# available. Otherwise, recover the canonical cached diagnostics produced by
# 4.1.1.3C so this block does not depend on a particular notebook run order.
if "support_diag" in globals():
    forecast_support_status = support_diag.copy()
elif diagnostics_path.exists():
    forecast_support_status = pd.read_parquet(diagnostics_path)
else:
    raise FileNotFoundError(
        "Forecast sequence support diagnostics are unavailable. "
        "Run 4.1.1.3C before applying the forecasting-support policy."
    )

# A Taxi Zone × Metric sequence is structurally available only when none of its
# five dayparts is classified as unavailable upstream. Structural absence is
# kept separate from intermittent missing observations within an otherwise
# valid mobility series.
forecast_support_status["structurally_available"] = (
    forecast_support_status["unavailable_dayparts"].eq(0)
)

# Apply the agreed 80% rule only to the post-CP development period already
# defined for the primary forecasting track.
#
# This is intentionally NOT a universal forecast-eligibility flag:
# 4.1.2 will apply the same minimum-support principle separately to the actual
# pre-CP and full-history training windows after their boundaries are defined.
forecast_support_status["post_cp_development_support_ok"] = (
    forecast_support_status["structurally_available"]
    & forecast_support_status["development_observed_share"].ge(
        MIN_FORECAST_TRAINING_COVERAGE
    )
)

# Preserve the reason for the current post-CP support result so downstream
# notebooks can distinguish structural absence from insufficient observed
# development history.
forecast_support_status["post_cp_support_reason"] = np.select(
    [
        ~forecast_support_status["structurally_available"],
        forecast_support_status["development_observed_share"].lt(
            MIN_FORECAST_TRAINING_COVERAGE
        ),
    ],
    [
        "structurally unavailable",
        "development coverage below 80%",
    ],
    default="meets development support rule",
)

# Mechanical contract validation: the policy layer must remain one record per
# canonical Taxi Zone × Metric sequence, with valid development coverage.
expected_sequence_count = (
    forecast_support_status[["taxi_zone_id", "metric"]]
    .drop_duplicates()
    .shape[0]
)

assert len(forecast_support_status) == expected_sequence_count
assert not forecast_support_status.duplicated(
    ["taxi_zone_id", "metric"]
).any()
assert forecast_support_status["development_observed_share"].between(
    0, 1, inclusive="both"
).all()

print(f"Forecast support policy rows: {len(forecast_support_status):,}")
print(
    "Minimum training-period coverage rule: "
    f"{MIN_FORECAST_TRAINING_COVERAGE:.0%}"
)

Forecast support policy rows: 1,315
Minimum training-period coverage rule: 80%


Summarize structural availability and post\-CP development support\. Show the immediate effect of the 80% support rule for the primary post\-CP development track, separating sequences that are structurally unavailable from those that simply lack sufficient observed development history\. This does not yet assign eligibility for the pre\-CP or full\-history modeling tracks\.

In [12]:
# 4.1.1.3G — Summarize structural availability and post-CP development support
#
# This is the reader-facing view of the support policy established above.
# It shows how many Taxi Zone × Metric sequences are structurally available
# and how many also satisfy the 80% observed-history requirement during the
# post-CP development period.
#
# Pre-CP and full-history eligibility are intentionally absent because their
# exact training windows will be defined in 4.1.2.

post_cp_support_summary = (
    forecast_support_status
    .groupby("metric", as_index=False)
    .agg(
        total_sequences=("taxi_zone_id", "size"),
        structurally_available=("structurally_available", "sum"),
        post_cp_support_ok=("post_cp_development_support_ok", "sum"),
    )
)

# Add shares so differences in support are easy to compare across targets.
post_cp_support_summary["structurally_available_share"] = (
    post_cp_support_summary["structurally_available"]
    / post_cp_support_summary["total_sequences"]
)

post_cp_support_summary["post_cp_support_ok_share"] = (
    post_cp_support_summary["post_cp_support_ok"]
    / post_cp_support_summary["total_sequences"]
)

display(
    post_cp_support_summary.style.format(
        {
            "structurally_available_share": "{:.3f}",
            "post_cp_support_ok_share": "{:.3f}",
        }
    )
)

,metric,total_sequences,structurally_available,post_cp_support_ok,structurally_available_share,post_cp_support_ok_share
0,fhvhv_avg_trip_speed,263,262,257,0.996,0.977
1,fhvhv_trip_count,263,263,258,1.000,0.981
2,subway_ridership,263,156,149,0.593,0.567
3,taxi_avg_trip_speed,263,262,201,0.996,0.764
4,taxi_trip_count,263,263,258,1.000,0.981


Findings\. The 80% post\-CP development\-support rule retains nearly all structurally available Taxi and FHVHV sequences: 258 of 263 Taxi trip\-count sequences, 258 of 263 FHVHV trip\-count sequences, and 257 of 262 FHVHV speed sequences qualify\. Among the 156 Taxi Zones with structurally available Subway ridership, 149 meet the support threshold, so Subway's lower overall qualifying share primarily reflects zones without Subway observations rather than poor coverage among available series\. Taxi average trip speed remains the main constraint: 201 of 262 structurally available sequences meet the 80% rule, leaving roughly one\-quarter of otherwise available Taxi\-speed series without sufficient post\-CP development history\. Overall, the threshold preserves almost the full forecasting population for four targets while applying meaningful filtering where Taxi\-speed coverage is genuinely thinner\.

> ✅ Decision — Forecasting support rule

A Taxi Zone × Metric sequence must have at least 80% observed coverage within the training period used by that forecasting track to qualify for modeling\. Sequences with structurally unavailable dayparts are excluded for that metric rather than treated as ordinary missing data\. For the primary post\-CP track, this rule is applied to the January 5, 2025–January 4, 2026 development period\. The same 80% rule will be applied separately to the pre\-CP and full\-history tracks once their training windows are defined in \`4\.1\.2\`\.

The earlier \`3\.3\.1\` \`modeled\` / \`fallback\` classifications are retained as provenance but do not determine forecasting eligibility\. A fallback sequence can still qualify when it has enough observed forecasting history\. Model families may add stricter continuity requirements later when needed\.

### Section Summary

Forecasting support is now defined at the Taxi Zone × Metric sequence level rather than inherited directly from the earlier 3\.3\.1 residual\-model routing\. All 1,315 forecasting sequences were matched to their upstream support provenance and evaluated for observed coverage, continuity, temporal extent, and pre/post\-CP availability\. The review showed that fallback status does not imply poor forecasting history, so it is retained as provenance rather than used as an exclusion rule\. Structural unavailability is handled separately, while otherwise available sequences use an 80% observed training\-period coverage threshold to qualify for forecasting\. Applied to the post\-CP development period, this rule retains nearly all Taxi trip, FHVHV trip, FHVHV speed, and available Subway sequences, while filtering a more meaningful share of Taxi speed sequences\. Pre\-CP and full\-history eligibility will be calculated separately in 4\.1\.2 using the same 80% rule once those training windows are defined\.

## 4.1.1.4 Construct and Validate Forecast Horizons

This chapter turns the ordered Taxi Zone × Metric sequences into actual forecasting targets. We will create three horizons: **1 step ahead**, **2 steps ahead**, and **5 steps ahead**. With five dayparts per day, `h=5` represents the same daypart on the following date. Each horizon keeps the future observation and its date, daypart, temporal bucket, and congestion-pricing period so later notebooks can assign modeling splits based on the value being predicted.

**Create the 1-step, 2-step, and 5-step forecast targets.** Shift each Taxi Zone × Metric sequence forward by the selected number of sequence positions and attach the future value and temporal fields to the current row. Missing observations stay in their original sequence positions, so a horizon always means the intended future daypart rather than the next available non-missing observation.

In [13]:
# 4.1.1.4A — Create the 1-step, 2-step, and 5-step forecast targets
#
# Every Taxi Zone × Metric sequence contains one row for each canonical
# Date × Daypart position, including positions where the observed mobility
# value is missing. Keeping those rows is essential: h=1 must always mean the
# next canonical daypart, not the next non-missing measurement.
#
# Selected horizons:
#   h=1 -> next sequence step
#   h=2 -> two sequence steps ahead
#   h=5 -> same daypart on the following date

FORECAST_HORIZONS = {
    1: "next sequence step",
    2: "two sequence steps ahead",
    5: "same daypart next day",
}

# Work from the ordered long-form forecasting layer created earlier in 4.1.1.
# Sort explicitly so every future-value shift follows the canonical sequence
# defined by observation_sequence_id within each Taxi Zone × Metric series.
forecast_horizon_panel = (
    forecast_sequence_long_df
    .sort_values(
        ["taxi_zone_id", "metric", "observation_sequence_id"]
    )
    .reset_index(drop=True)
    .copy()
)

sequence_group = forecast_horizon_panel.groupby(
    ["taxi_zone_id", "metric"],
    sort=False,
)

# Carry the future observed value plus the temporal information needed to
# identify exactly what is being predicted. Target pre/post-CP status is also
# retained because a forecast origin can occur before the policy boundary
# while its future target falls after it.
target_fields = [
    "observed_value",
    "observation_sequence_id",
    "date",
    "daypart",
    "daypart_order",
    "temporal_bucket",
    "temporal_bucket_order",
    "pre_post_cp",
]

for horizon in FORECAST_HORIZONS:
    shifted_targets = sequence_group[target_fields].shift(-horizon)

    shifted_targets.columns = [
        f"target_{column}_h{horizon}"
        for column in target_fields
    ]

    forecast_horizon_panel = pd.concat(
        [forecast_horizon_panel, shifted_targets],
        axis=1,
    )

print(f"Forecast-origin rows: {len(forecast_horizon_panel):,}")
print(f"Taxi Zone × Metric sequences: {forecast_horizon_panel['forecast_sequence_id'].nunique():,}")
print(f"Forecast horizons: {list(FORECAST_HORIZONS)}")

Forecast-origin rows: 7,797,950
Taxi Zone × Metric sequences: 1,315
Forecast horizons: [1, 2, 5]


**Check that each horizon lands on the correct future sequence position, daypart, and date.** Validate the shifted targets against the five-daypart cycle, including transitions that cross into a new date or cross the congestion-pricing boundary. The only rows allowed to lack target metadata are the final `h` positions at the end of each Taxi Zone × Metric sequence.

In [14]:
# 4.1.1.4B — Check forecast-horizon alignment
#
# Horizon construction is valid only if the shifted target remains aligned
# with the canonical sequence:
#   1) target observation_sequence_id = origin observation_sequence_id + h
#   2) target daypart follows the five-daypart cycle
#   3) target date advances whenever the shift crosses a daily boundary
#
# Rows at the end of a sequence cannot have a future target because the study
# period ends first. Those expected boundary rows are tracked separately.

# Build the canonical five-daypart cycle directly from the ordered source.
daypart_cycle = (
    forecast_horizon_panel[
        ["daypart_order", "daypart"]
    ]
    .drop_duplicates()
    .sort_values("daypart_order")
    .reset_index(drop=True)
)

assert len(daypart_cycle) == 5, (
    "Expected exactly five canonical dayparts."
)

daypart_orders = daypart_cycle["daypart_order"].to_numpy()

# Convert each stored daypart_order into a zero-based cycle position so the
# validation does not depend on the numeric values used by the upstream field.
daypart_index_map = {
    order_value: cycle_index
    for cycle_index, order_value in enumerate(daypart_orders)
}

origin_daypart_index = (
    forecast_horizon_panel["daypart_order"]
    .map(daypart_index_map)
)

sequence_count = (
    forecast_horizon_panel["forecast_sequence_id"]
    .nunique()
)

validation_rows = []

for horizon, horizon_label in FORECAST_HORIZONS.items():

    target_sequence_col = (
        f"target_observation_sequence_id_h{horizon}"
    )
    target_date_col = f"target_date_h{horizon}"
    target_daypart_order_col = (
        f"target_daypart_order_h{horizon}"
    )
    target_period_col = (
        f"target_pre_post_cp_h{horizon}"
    )

    # A future target exists for every row except the final h positions within
    # each Taxi Zone × Metric sequence.
    has_future_target = (
        forecast_horizon_panel[target_sequence_col].notna()
    )

    check_df = forecast_horizon_panel.loc[
        has_future_target
    ].copy()

    check_origin_daypart_index = (
        origin_daypart_index.loc[has_future_target]
    )

    # Moving forward h sequence steps can change the daypart and cross one or
    # more calendar-day boundaries.
    expected_target_cycle_index = (
        check_origin_daypart_index + horizon
    ) % len(daypart_orders)

    expected_date_offset = (
        check_origin_daypart_index + horizon
    ) // len(daypart_orders)

    expected_target_daypart_order = daypart_orders[
        expected_target_cycle_index.astype(int)
    ]

    expected_target_date = (
        pd.to_datetime(check_df["date"])
        + pd.to_timedelta(expected_date_offset, unit="D")
    )

    actual_target_date = pd.to_datetime(
        check_df[target_date_col]
    )

    # Count forecast origins whose future target crosses the CP boundary.
    # These are valid targets and will matter when 4.1.2 assigns modeling
    # splits according to the target timestamp.
    regime_crossings = int(
        (
            check_df["pre_post_cp"]
            != check_df[target_period_col]
        ).sum()
    )

    actual_end_rows = int(
        (~has_future_target).sum()
    )

    expected_end_rows = (
        horizon * sequence_count
    )

    validation_rows.append(
        {
            "horizon": horizon,
            "horizon_label": horizon_label,
            "rows_checked": len(check_df),
            "sequence_position_mismatches": int(
                (
                    check_df[target_sequence_col]
                    != (
                        check_df["observation_sequence_id"]
                        + horizon
                    )
                ).sum()
            ),
            "daypart_order_mismatches": int(
                (
                    check_df[target_daypart_order_col]
                    != expected_target_daypart_order
                ).sum()
            ),
            "target_date_mismatches": int(
                (
                    actual_target_date
                    != expected_target_date
                ).sum()
            ),
            "regime_crossing_origins": regime_crossings,
            "end_rows_without_target": actual_end_rows,
            "expected_end_rows": expected_end_rows,
        }
    )

forecast_horizon_validation = pd.DataFrame(
    validation_rows
)

display(forecast_horizon_validation)

,horizon,horizon_label,rows_checked,sequence_position_mismatches,daypart_order_mismatches,target_date_mismatches,regime_crossing_origins,end_rows_without_target,expected_end_rows
0,1,next sequence step,7796635,0,0,0,1315,1315,1315
1,2,two sequence steps ahead,7795320,0,0,0,2630,2630,2630
2,5,same daypart next day,7791375,0,0,0,6575,6575,6575


**Findings.** All three forecast horizons align cleanly with the ordered mobility sequences. The h=1, h=2, and h=5 targets advance by exactly 1, 2, and 5 observation_sequence_id positions, with no daypart-order or target-date mismatches. The expected end-of-series rows are the only rows without future targets, confirming that missing horizon values at the end of March 2026 come from the study boundary rather than broken sequence construction. Forecast origins that cross the January 5, 2025 congestion-pricing boundary are also retained correctly, allowing later split logic to assign each observation according to the period of the target being predicted.

**Check how much observed target data remains at each horizon.** Count the valid forecast origins and observed future values for every metric and horizon. A missing future mobility value does not mean the horizon is broken—it means the correct target position exists but that measurement was not observed.

In [15]:
# 4.1.1.4C — Check observed target coverage by metric and horizon
#
# Horizon construction preserves source missingness. The coverage calculation
# therefore distinguishes:
#   - no future sequence position because the study period ends, from
#   - a valid future position whose mobility value is genuinely missing.
#
# End-of-sequence rows are excluded from the denominator because they are not
# valid forecast origins for that horizon.

target_coverage_rows = []

for horizon, horizon_label in FORECAST_HORIZONS.items():

    target_sequence_col = (
        f"target_observation_sequence_id_h{horizon}"
    )
    target_value_col = (
        f"target_observed_value_h{horizon}"
    )

    horizon_rows = forecast_horizon_panel.loc[
        forecast_horizon_panel[target_sequence_col].notna()
    ]

    for metric, metric_df in horizon_rows.groupby(
        "metric",
        sort=True,
    ):
        observed_targets = int(
            metric_df[target_value_col].notna().sum()
        )

        target_coverage_rows.append(
            {
                "metric": metric,
                "horizon": horizon,
                "horizon_label": horizon_label,
                "available_forecast_origins": len(metric_df),
                "observed_targets": observed_targets,
                "missing_targets": (
                    len(metric_df) - observed_targets
                ),
                "observed_target_share": (
                    observed_targets / len(metric_df)
                ),
            }
        )

forecast_horizon_target_coverage = pd.DataFrame(
    target_coverage_rows
)

display(
    forecast_horizon_target_coverage.style.format(
        {
            "observed_target_share": "{:.3f}",
        }
    )
)

,metric,horizon,horizon_label,available_forecast_origins,observed_targets,missing_targets,observed_target_share
0,fhvhv_avg_trip_speed,1,next sequence step,1559327,1523215,36112,0.977
1,fhvhv_trip_count,1,next sequence step,1559327,1531245,28082,0.982
2,subway_ridership,1,next sequence step,1559327,912057,647270,0.585
3,taxi_avg_trip_speed,1,next sequence step,1559327,1124962,434365,0.721
4,taxi_trip_count,1,next sequence step,1559327,1531245,28082,0.982
5,fhvhv_avg_trip_speed,2,two sequence steps ahead,1559064,1522960,36104,0.977
6,fhvhv_trip_count,2,two sequence steps ahead,1559064,1530989,28075,0.982
7,subway_ridership,2,two sequence steps ahead,1559064,911901,647163,0.585
8,taxi_avg_trip_speed,2,two sequence steps ahead,1559064,1124809,434255,0.721
9,taxi_trip_count,2,two sequence steps ahead,1559064,1530989,28075,0.982


**Findings.** Observed target coverage remains consistent across the three forecast horizons and continues to reflect the underlying availability of each mobility metric rather than the horizon construction itself. Taxi and FHVHV trip counts retain the strongest target coverage, followed by FHVHV speed; Taxi speed remains more fragmented, and Subway coverage remains limited to zones with observed Subway service. Moving from h=1 to h=2 and h=5 does not introduce a meaningful new loss of target data beyond the expected final sequence positions, so all three horizons can move forward into split construction and modeling.

### Section Summary

The ordered mobility sequences now support three validated forecasting horizons: the next daypart \(h=1\), two dayparts ahead \(h=2\), and the same daypart on the following date \(h=5\)\. Each future target remains tied to its exact sequence position, date, daypart, temporal bucket, and pre/post\-CP period, while genuine source missingness is preserved rather than skipped\. These checks confirm that the horizon layer is ready for 4\.1\.2, where train, validation, and test assignments can be based on the timestamp of the future value being predicted\.

---

## 4.1.1.5 Package the Forecasting Target Handoff

This chapter packages the sequence, horizon, and support information into a small set of reusable forecasting inputs. The main sequence panel keeps one row per Taxi Zone × Metric × sequence position with the three future targets attached. A separate support registry keeps the sequence-level coverage and eligibility information without repeating it millions of times, while a three-row horizon registry records the forecast horizons used throughout the forecasting workflow.

**Build the sequence-level support registry.** Keep one record per Taxi Zone × Metric with the support diagnostics created earlier, including structural availability and the 80% post-CP development-support result. This avoids repeating the same support information on every row of the much larger forecasting sequence panel.

In [16]:
# 4.1.1.5A — Build the sequence-level support registry
#
# Support is a property of an entire Taxi Zone × Metric sequence rather than
# an individual forecast-origin row. Keep those diagnostics in a separate
# registry so downstream notebooks can join them only when needed.
#
# The current post-CP support flag is preserved here, while 4.1.2 will add
# separate pre-CP and full-history eligibility fields after those training
# windows are defined.

support_registry_columns = [
    "taxi_zone_id",
    "zone",
    "borough",
    "metric",
    "forecast_sequence_id",
    "total_sequence_rows",
    "non_null_observed_rows",
    "overall_coverage_rate",
    "first_non_null_sequence_position",
    "last_non_null_sequence_position",
    "first_non_null_timestamp",
    "last_non_null_timestamp",
    "active_sequence_span",
    "longest_consecutive_observed_run",
    "missing_run_count",
    "max_missing_run_length",
    "pre_cp_possible_rows",
    "pre_cp_observed_rows",
    "pre_cp_observed_share",
    "post_cp_possible_rows",
    "post_cp_observed_rows",
    "post_cp_observed_share",
    "development_possible_rows",
    "development_observed_rows",
    "development_observed_share",
    "evaluation_possible_rows",
    "evaluation_observed_rows",
    "evaluation_observed_share",
    "modeled_dayparts",
    "fallback_dayparts",
    "unavailable_dayparts",
    "represented_support_dayparts",
    "support_pattern",
    "structurally_available",
    "post_cp_development_support_ok",
    "post_cp_support_reason",
]

forecast_sequence_support_registry = (
    forecast_support_status[
        support_registry_columns
    ]
    .sort_values(
        ["metric", "taxi_zone_id"]
    )
    .reset_index(drop=True)
    .copy()
)

# The support registry must contain exactly one row per Taxi Zone × Metric
# forecasting sequence.
assert not forecast_sequence_support_registry.duplicated(
    ["taxi_zone_id", "metric"]
).any()

assert (
    forecast_sequence_support_registry[
        "forecast_sequence_id"
    ].nunique()
    == len(forecast_sequence_support_registry)
)

print(
    "Support registry:",
    f"{len(forecast_sequence_support_registry):,} sequences × "
    f"{forecast_sequence_support_registry.shape[1]} fields"
)

Support registry: 1,315 sequences × 36 fields


**Build the reusable horizon registry.** Store the three approved forecast horizons in a compact table so downstream notebooks do not need to redefine what `h=1`, `h=2`, and `h=5` mean.

In [17]:
# 4.1.1.5B — Build the reusable horizon registry
#
# This small table is executable metadata: downstream split, feature, baseline,
# and model notebooks can load the same horizon definitions rather than
# hard-coding them independently.

forecast_horizon_registry = pd.DataFrame(
    [
        {
            "horizon": horizon,
            "horizon_label": horizon_label,
            "sequence_steps_ahead": horizon,
            "dayparts_per_day": 5,
            "same_daypart_next_day": horizon == 5,
        }
        for horizon, horizon_label in FORECAST_HORIZONS.items()
    ]
).sort_values("horizon").reset_index(drop=True)

assert (
    forecast_horizon_registry["horizon"].tolist()
    == [1, 2, 5]
)

assert forecast_horizon_registry["horizon"].is_unique

display(forecast_horizon_registry)

,horizon,horizon_label,sequence_steps_ahead,dayparts_per_day,same_daypart_next_day
0,1,next sequence step,1,5,False
1,2,two sequence steps ahead,2,5,False
2,5,same daypart next day,5,5,True


**Validate the three forecasting handoff tables.** Confirm that the sequence panel still has one row per Taxi Zone × Metric × observation sequence position, the support registry has one row per Taxi Zone × Metric, and the horizon registry contains only the three selected horizons. Also confirm that every populated target position remains exactly the requested number of sequence steps ahead.

In [18]:
# 4.1.1.5C — Validate the forecasting handoff tables
#
# The three outputs serve different downstream jobs:
#   sequence panel   -> ordered observations and future targets
#   support registry -> sequence-level support and eligibility information
#   horizon registry -> shared definitions of h=1, h=2, and h=5
#
# Validate each table at its intended grain before any files are exported.

sequence_key = [
    "taxi_zone_id",
    "metric",
    "observation_sequence_id",
]

support_key = [
    "taxi_zone_id",
    "metric",
]

sequence_duplicate_count = int(
    forecast_horizon_panel.duplicated(
        sequence_key
    ).sum()
)

support_duplicate_count = int(
    forecast_sequence_support_registry
    .duplicated(support_key)
    .sum()
)

horizon_alignment_errors = 0

for horizon in FORECAST_HORIZONS:

    target_sequence_col = (
        f"target_observation_sequence_id_h{horizon}"
    )

    populated_targets = (
        forecast_horizon_panel[
            target_sequence_col
        ].notna()
    )

    horizon_alignment_errors += int(
        (
            forecast_horizon_panel.loc[
                populated_targets,
                target_sequence_col,
            ]
            != (
                forecast_horizon_panel.loc[
                    populated_targets,
                    "observation_sequence_id",
                ]
                + horizon
            )
        ).sum()
    )

handoff_validation = pd.DataFrame(
    [
        {
            "output": "Forecast sequence panel",
            "rows": len(forecast_horizon_panel),
            "unique_keys": (
                forecast_horizon_panel[
                    sequence_key
                ]
                .drop_duplicates()
                .shape[0]
            ),
            "duplicate_keys": sequence_duplicate_count,
        },
        {
            "output": "Sequence support registry",
            "rows": len(
                forecast_sequence_support_registry
            ),
            "unique_keys": (
                forecast_sequence_support_registry[
                    support_key
                ]
                .drop_duplicates()
                .shape[0]
            ),
            "duplicate_keys": support_duplicate_count,
        },
        {
            "output": "Forecast horizon registry",
            "rows": len(
                forecast_horizon_registry
            ),
            "unique_keys": (
                forecast_horizon_registry[
                    ["horizon"]
                ]
                .drop_duplicates()
                .shape[0]
            ),
            "duplicate_keys": int(
                forecast_horizon_registry[
                    "horizon"
                ].duplicated().sum()
            ),
        },
    ]
)

assert sequence_duplicate_count == 0
assert support_duplicate_count == 0
assert horizon_alignment_errors == 0
assert (
    set(forecast_horizon_registry["horizon"])
    == {1, 2, 5}
)

display(handoff_validation)

,output,rows,unique_keys,duplicate_keys
0,Forecast sequence panel,7797950,7797950,0
1,Sequence support registry,1315,1315,0
2,Forecast horizon registry,3,3,0


**Findings.** The forecasting handoff preserves the intended grain of all three outputs. The main sequence panel contains 7,797,950 unique Taxi Zone × Metric × observation-sequence records with no duplicate keys, the support registry contains one record for each of the 1,315 Taxi Zone × Metric sequences with no duplicates, and the horizon registry contains exactly the three approved horizons. All populated future target positions remain exactly the requested number of sequence steps ahead, confirming that packaging the outputs did not alter the target alignment established in the previous section.

### Section Summary

The forecasting inputs are now separated into three focused handoff tables\. The sequence panel carries the full ordered mobility history and future targets, the support registry carries one set of coverage and support diagnostics per Taxi Zone × Metric sequence, and the horizon registry provides one shared definition of h=1, h=2, and h=5 for downstream notebooks\. Keeping these concerns separate avoids repeating sequence\-level metadata across millions of observations while giving later forecasting notebooks a consistent set of targets, support information, and horizon definitions\.

---

## 4.1.1.6 Export Forecasting Target Sequences and Horizons

This final chapter writes the validated forecasting inputs to the `4.1.1.final_tables` layer. The exported sequence panel carries the ordered observations and all three forecast targets, the support registry carries sequence-level support information, and the horizon registry provides the shared horizon definitions needed by the split, feature-engineering, and modeling notebooks that follow.

**Export the forecasting handoff tables.** Write the sequence panel, support registry, and horizon registry as compressed Parquet files under the `4.1.1.final_tables` directory.

In [19]:
# 4.1.1.6A — Export the forecasting handoff tables
#
# These files are the authoritative outputs of 4.1.1:
#
#   forecasting_target_sequence_panel.parquet
#       One row per Taxi Zone × Metric × observation sequence position with
#       h=1, h=2, and h=5 future targets attached.
#
#   forecasting_sequence_support_registry.parquet
#       One row per Taxi Zone × Metric with coverage, continuity, upstream
#       support provenance, structural availability, and the current post-CP
#       development-support result.
#
#   forecast_horizon_registry.parquet
#       Three reusable horizon definitions shared by downstream notebooks.

forecast_output_dir = Path(
    "pipeline_data/4.1.1.final_tables"
)

forecast_output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

sequence_output_path = (
    forecast_output_dir
    / "forecasting_target_sequence_panel.parquet"
)

support_output_path = (
    forecast_output_dir
    / "forecasting_sequence_support_registry.parquet"
)

horizon_output_path = (
    forecast_output_dir
    / "forecast_horizon_registry.parquet"
)

forecast_horizon_panel.to_parquet(
    sequence_output_path,
    index=False,
    compression="zstd",
)

forecast_sequence_support_registry.to_parquet(
    support_output_path,
    index=False,
    compression="zstd",
)

forecast_horizon_registry.to_parquet(
    horizon_output_path,
    index=False,
    compression="zstd",
)

**Confirm the exported files and their sizes.** Check that all three final Parquet files exist and report their row counts, column counts, and disk sizes so the notebook ends with a compact record of exactly what was handed downstream.

In [20]:
# 4.1.1.6B — Confirm the exported forecasting files
#
# Keep the final export check compact. Verify that each file exists and record
# its in-memory dimensions and on-disk size without printing large previews.

export_manifest = pd.DataFrame(
    [
        {
            "output": "Forecast sequence panel",
            "path": str(sequence_output_path),
            "rows": len(
                forecast_horizon_panel
            ),
            "columns": (
                forecast_horizon_panel.shape[1]
            ),
            "exists": (
                sequence_output_path.exists()
            ),
            "size_mb": (
                sequence_output_path.stat().st_size
                / 1024**2
            ),
        },
        {
            "output": "Sequence support registry",
            "path": str(support_output_path),
            "rows": len(
                forecast_sequence_support_registry
            ),
            "columns": (
                forecast_sequence_support_registry
                .shape[1]
            ),
            "exists": (
                support_output_path.exists()
            ),
            "size_mb": (
                support_output_path.stat().st_size
                / 1024**2
            ),
        },
        {
            "output": "Forecast horizon registry",
            "path": str(horizon_output_path),
            "rows": len(
                forecast_horizon_registry
            ),
            "columns": (
                forecast_horizon_registry.shape[1]
            ),
            "exists": (
                horizon_output_path.exists()
            ),
            "size_mb": (
                horizon_output_path.stat().st_size
                / 1024**2
            ),
        },
    ]
)

assert export_manifest["exists"].all()

display(
    export_manifest.style.format(
        {
            "size_mb": "{:.2f}",
        }
    )
)

,output,path,rows,columns,exists,size_mb
0,Forecast sequence panel,pipeline_data/4.1.1.final_tables/forecasting_target_sequence_panel.parquet,7797950,38,True,119.56
1,Sequence support registry,pipeline_data/4.1.1.final_tables/forecasting_sequence_support_registry.parquet,1315,36,True,0.06
2,Forecast horizon registry,pipeline_data/4.1.1.final_tables/forecast_horizon_registry.parquet,3,5,True,0.00


## Close

\`4\.1\.1\` establishes the forecasting target layer used throughout Chapter 4\. Five mobility targets are represented as ordered Taxi Zone × Metric sequences that preserve the full five\-daypart daily cycle and genuine missing observations\. Forecasting support is evaluated separately from the earlier residual\-model routing, with structural availability distinguished from ordinary missingness and an 80% minimum training\-coverage rule established for later track\-specific eligibility\. The final handoff provides 1\-step, 2\-step, and 5\-step future targets together with their future temporal and policy\-regime metadata, allowing \`4\.1\.2\` to construct chronological modeling splits using the timestamp being predicted rather than the forecast origin\.

### 4.1.1 Final Inventory of Files

| File                                            | Description                                                                                                                                                                |
| ----------------------------------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `forecasting_target_sequence_panel.parquet`     | Main forecasting target layer at the Taxi Zone × Metric × observation sequence position grain, with `h=1`, `h=2`, and `h=5` future targets and their target-time metadata. |
| `forecasting_sequence_support_registry.parquet` | One row per Taxi Zone × Metric containing coverage, continuity, support provenance, structural availability, and the post-CP 80% support result.                           |
| `forecast_horizon_registry.parquet`             | Defines the three shared forecasting horizons: `h=1`, `h=2`, and `h=5`.                                                                                                    |
| `forecast_sequence_support_diagnostics.parquet` | Intermediate sequence-level diagnostic table used to evaluate coverage, continuity, missingness, and forecasting support before the final eligibility rule was set.        |


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=5c43663c-345d-4c21-9332-dd4f928af3ba' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>